In [1]:
# JSON + subDescription 적용
# python에서 json으로 데이터를 받아서 필드 검색

import requests
import pandas as pd
from tqdm.notebook import tqdm
import time

In [ ]:
"""
_SERVICE_KEY = "07251ecd-1b8b-4cad-9205-ca69f48c0df3"
URL = "https://api.kcisa.kr/openapi/service/rest/meta13/getCTE01701"

def test_api_with_json():
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": 10,
        "pageNo": 1
    }

    # header에 json으로 받겠다고 명시
    headers = {
        "Accept": "application/json"
    }

    response = requests.get(URL, params=params, headers=headers)

    if response.status_code == 200:
        data = response.json()
        # json 구조 파고들어가서 subDescription 확인
        try:
            items = data["response"]["body"]["items"]["item"]
            print(f"첫 번째 아이템 정보:")
            print(f"단어: {items[0]["title"]}")
            print(f"상세페이지(url): {items[0].get("url")}")
            print(f"실제 영상 주소(subDescription): {items[0].get("subDescription")}")
            return items
        except KeyError as e:
            print(f"json 구조가 예상과 다름: {e}")
            # 실제 구조 출력해서 확인
            print(data)
    else:
        print(f"호출 실패: {response.status_code}")

test_items = test_api_with_json()
"""

In [ ]:
# 실제 영상 주소(subDescription): http://sldict.korean.go.kr/multimedia/multimedia_files/convert/20160325/281631/MOV000277619_700X466.mp4
# 영상 주소 가져오면서 평탄화 진행

_SERVICE_KEY = "07251ecd-1b8b-4cad-9205-ca69f48c0df3"
URL = "https://api.kcisa.kr/openapi/service/rest/meta13/getCTE01701"

In [ ]:
all_data = []
# 일상생활 수어 총 351 페이지
total_pages = 351

print("데이터를 json 형식으로 수집 및 평탄화 시작...")

for page in tqdm(range(1, total_pages + 1)):
    params = {
        "serviceKey": _SERVICE_KEY,
        "numOfRows": 100,
        "pageNo": page
    }
    headers = {"Accept": "application/json"}

    try:
        response = requests.get(URL, params=params, headers=headers, timeout=15)
        if response.status_code == 200:
            data = response.json()
            # items 상자를 꺼내기 전 안전장치 추가
            body = data.get("response", {}).get("body", {})
            items_container = body.get("items")

            # items_container가 None 이거나 비어있지 않은지 체크
            if items_container and "item" in items_container:
                items = items_container["item"]

                # 리스트 형태가 아닐 경우 (데이터가 1개일 때) 처리
                if isinstance(items, dict):
                    items = [items]

            # items = data["response"]["body"]["items"]["item"]

                for item in items:
                    # 한국어 대응 표현 (예: "득하다, 획득")
                    raw_words = item.get("title", "")
                    mp4_url = item.get("subDescription")
                    # 상세페이지 주소 (보험용)
                    detail_url = item.get("url")

                    # 평탄화 작업 진행: "득하다, 획득" -> ["득하다", "획득"]
                    if raw_words:
                        # 쉼표로 쪼개고 앞뒤 공백 제거
                        word_list = [w.strip() for w in raw_words.split(",") if w.strip()]

                        for word in word_list:
                            all_data.append({
                                "word": word,
                                "video_url": mp4_url,
                                # 참고용 원본 뭉치 보관
                                "origin_word": raw_words,
                                "detail_url": detail_url
                            })
            else:
                print(f"{page}페이지 호출 실패: {response.status_code}")
                # 더 이상 가져올 데이터가 없으면 루프 탈출
                print(f"{page}페이지: 더 이상 데이터가 없습니다. (수집 종료)")
                break

    except Exception as e:
        print(f"{page}페이지 예외 발생: {e}")

    time.sleep(0.2)

데이터를 json 형식으로 수집 및 평탄화 시작...


  0%|          | 0/351 [00:00<?, ?it/s]

39페이지 호출 실패: 200
39페이지: 더 이상 데이터가 없습니다. (수집 종료)


In [5]:
print(f"현재까지 수집된 행(단어) 개수: {len(all_data)}")
if all_data:
    print("마지막 수집 단어 예시:", all_data[-1]['word'])

현재까지 수집된 행(단어) 개수: 7554
마지막 수집 단어 예시: 접시


In [6]:
# 데이터프레임 변환 및 저장
df_final = pd.DataFrame(all_data)
df_final.to_csv("sign_mp4_url.csv", index=False, encoding='utf-8-sig')

print(f"\n 수집 완료! 총 {len(df_final)}개의 단어-영상 매칭 데이터가 생성됨")


 수집 완료! 총 7554개의 단어-영상 매칭 데이터가 생성됨
